# prompt-recipe-smith Colab Demo

This notebook demonstrates the core `prompt-recipe-smith` flow:

1. Start with a rough idea.
2. Ask three clarification questions.
3. Generate a final prompt for a chat agent.

The built-in questions are in English, but your rough idea and answers can be in any language.

## Install

In Google Colab, run the install cell below before importing the library. It installs the latest repository version so the layered session API is available.

In [ ]:
# Colab install for the latest repository version
%pip install -q --upgrade --force-reinstall git+https://github.com/yeiichi/prompt-recipe-smith.git

In [ ]:
# Optional PyPI install, for released versions 0.2.0 and newer
# %pip install -q --upgrade "prompt-recipe-smith>=0.2.0"

## Import the Library

In [ ]:
import importlib.metadata as metadata
import inspect
import sys

for module_name in list(sys.modules):
    if module_name == "prompt_recipe_smith" or module_name.startswith("prompt_recipe_smith."):
        del sys.modules[module_name]

from prompt_recipe_smith import PromptBuilder
from prompt_recipe_smith.output import to_json, to_plain_text
from prompt_recipe_smith.recipes import clarify_idea_recipe
from prompt_recipe_smith.session import PromptSessionRunner

installed_version = metadata.version("prompt-recipe-smith")
loaded_from = inspect.getfile(PromptSessionRunner)
if not hasattr(PromptSessionRunner, "start"):
    raise RuntimeError(
        "This notebook needs prompt-recipe-smith 0.2.0 or newer. "
        f"Loaded version {installed_version} from {loaded_from}. "
        "Run the GitHub install cell above, then rerun this import cell."
    )

print(f"prompt-recipe-smith {installed_version} loaded from {loaded_from}")

## One-shot Prompt Build

The one-shot builder renders a prompt immediately from the rough idea. This is useful for quick checks and non-interactive workflows.

In [ ]:
recipe = clarify_idea_recipe()
result = PromptBuilder().build(recipe, "I want to plan a small workshop")

print(to_plain_text(result))

## Three-question Layered Flow

The layered flow mirrors the CLI behavior. It asks three clarification questions and then generates a final prompt for the chat agent.

In [ ]:
runner = PromptSessionRunner(
    recipe=clarify_idea_recipe(),
    builder=PromptBuilder(),
)

session = runner.start("I want to plan a small workshop")

while not session.complete and session.next_question is not None:
    answer = input(f"{session.next_question.text}\n> ")
    session = runner.answer(session, answer)

layered_result = runner.finish(session)
print(to_plain_text(layered_result))

## Multilingual Answers

The questions are English, but the rough idea and answers can use Unicode text. This example uses Japanese answers.

In [ ]:
runner = PromptSessionRunner(
    recipe=clarify_idea_recipe(),
    builder=PromptBuilder(),
)

session = runner.start("ワークショップを計画したい")
session = runner.answer(session, "議題を整理する")
session = runner.answer(session, "新任マネージャー向け")
session = runner.answer(session, "日本語で簡潔に")

japanese_result = runner.finish(session)
print(to_plain_text(japanese_result))

## JSON Output

Use JSON output when another tool or app needs structured data. Unicode text remains readable.

In [ ]:
if "japanese_result" not in globals():
    runner = PromptSessionRunner(
        recipe=clarify_idea_recipe(),
        builder=PromptBuilder(),
    )
    session = runner.start("ワークショップを計画したい")
    session = runner.answer(session, "議題を整理する")
    session = runner.answer(session, "新任マネージャー向け")
    session = runner.answer(session, "日本語で簡潔に")
    japanese_result = runner.finish(session)

print(to_json(japanese_result))

## Notes

- `PromptSessionRunner` manages the three-question flow.
- `PromptBuilder` renders the final prompt.
- Built-in branch keywords are defined in `src/prompt_recipe_smith/recipes/builtin.py`.
- Keyword matching is handled by `PromptBranch.matches()` in `src/prompt_recipe_smith/models.py`.
- The package does not call external LLM APIs; it generates the prompt text locally.